In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import vectorbt as vbt
from datetime import datetime, timedelta
import warnings
import contextlib
import tqdm as tqdm
from tabulate import tabulate
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from datetime import datetime
import hashlib
import itertools
import os
import json 
import mplfinance as mpf
warnings.filterwarnings('ignore') 

In [2]:
data_path = r"C:\Users\OMEN\Desktop\New folder (2)\data\nifty500\nifty500_daily_ohlcv.csv"
data = pd.read_csv(data_path, header=[0,1], index_col=0, parse_dates=True)
# 1. Convert index to Datetime
data.index = pd.to_datetime(data.index)

# 2. Drop any rows where the date is missing (prevents KeyError: NaT)
data = data.loc[data.index.notnull()]

# 3. Sort index (required for rolling/shifting logic)
data = data.sort_index()


In [3]:
def calculate_donchian(df: pd.DataFrame, window: int):
    """Flexible Donchian Channel calculation."""
    # Find columns regardless of case
    cols = {col.lower(): col for col in df.columns}
    high_col, low_col = cols.get('high'), cols.get('low')
    
    # Shift(1) is vital to avoid lookahead bias
    upper = df[high_col].rolling(window=window).max().shift(1)
    lower = df[low_col].rolling(window=window).min().shift(1)
    return upper, lower, (upper + lower) / 2

In [4]:
def calculate_hurst(ts, window=100):
    """Fast Vectorized Hurst Exponent using NumPy sliding windows."""
    vals = ts.values
    n = len(vals)
    hurst_out = np.full(n, 0.5) # Default to 0.5
    
    if n < window:
        return pd.Series(hurst_out, index=ts.index)

    lags = np.arange(2, 20)
    log_lags = np.log(lags)

    # Create sliding window view (memory efficient)
    shape = (n - window + 1, window)
    strides = (vals.strides[0], vals.strides[0])
    windows = np.lib.stride_tricks.as_strided(vals, shape=shape, strides=strides)

    results = []
    # Loop over windows, but perform math in optimized NumPy
    for chunk in windows:
        # Vectorized standard deviation of differences for all lags
        tau = [np.sqrt(np.std(np.subtract(chunk[lag:], chunk[:-lag]))) for lag in lags]
        poly = np.polyfit(log_lags, np.log(tau), 1)
        results.append(poly[0] * 2.0)
    
    hurst_out[window-1:] = results
    return pd.Series(hurst_out, index=ts.index)

In [5]:
def signals_to_sticky_weights(signals: pd.DataFrame, max_stocks: int) -> pd.DataFrame:
    """
    Convert entry/exit signals into sticky portfolio weights.

    Contract:
    - signal[t] is EXECUTED at OPEN[t]
    - +1 = enter
    - -1 = exit
    - 0  = hold
    """

    weights = pd.DataFrame(
        np.nan, 
        index=signals.index,
        columns=signals.columns
    )

    current_weights = pd.Series(0.0, index=signals.columns)
    free_pool = 1.0

    for date in signals.index:
        day_signal = signals.loc[date]

        # ================= SELL FIRST =================
        sell_stocks = day_signal[day_signal == -1].index

        for stock in sell_stocks:
            if current_weights[stock] > 0:
                free_pool += current_weights[stock]
                current_weights[stock] = 0.0

        # ================= SLOT CALC =================
        current_holdings = (current_weights > 0).sum()
        # FIXED: Use the parameter 'max_stocks' instead of global 'MAX_STOCKS'
        available_slots = max(0, max_stocks - current_holdings)

        # ================= BUY =================
        buy_stocks = day_signal[day_signal == 1].index.tolist()

        if available_slots > 0 and free_pool > 0 and len(buy_stocks) > 0:
            selected = buy_stocks[:available_slots]
            allocation = free_pool / len(selected)

            for stock in selected:
                current_weights[stock] = allocation

            free_pool = 0.0

        # ================= SAVE STATE =================
        day_weights = pd.Series(np.nan, index=signals.columns)

        # Entry days
        entry_stocks = day_signal[day_signal == 1].index
        day_weights[entry_stocks] = current_weights[entry_stocks]

        # Exit days
        exit_stocks = day_signal[day_signal == -1].index
        day_weights[exit_stocks] = 0.0

        # HOLD days intentionally left as NaN
        weights.loc[date] = day_weights

    return weights

In [6]:
class Strategy:
    _indicators = None
    _params     = None
    _weights_df = None
    _signals_df = None

    @classmethod
    def prepare(cls, data, params):
        cls._params = params
        tickers = data.columns.get_level_values(0).unique()
        
        # 1. INITIALIZE INDICATOR STORAGE
        cls._indicators = {
            "UP_ENTRY": pd.DataFrame(index=data.index, columns=tickers),
            "LO_ENTRY": pd.DataFrame(index=data.index, columns=tickers),
            "UP_EXIT":  pd.DataFrame(index=data.index, columns=tickers),
            "LO_EXIT":  pd.DataFrame(index=data.index, columns=tickers),
            "HURST":    pd.DataFrame(index=data.index, columns=tickers)
        }

        # 2. PRECOMPUTE INDICATORS (🧬 1/3: Indicators)
        for t in tqdm.tqdm(tickers, desc="🧬 1/3: Indicators", leave=True):
            df_t = data[t]
            c_col = next(c for c in df_t.columns if c.lower() == 'close')
            
            u_e, l_e, _ = calculate_donchian(df_t, params["LOOKBACK"])
            u_x, l_x, _ = calculate_donchian(df_t, params["EXIT_LOOKBACK"])
            h_vals = calculate_hurst(df_t[c_col], window=params["HURST_WINDOW"])
            
            cls._indicators["UP_ENTRY"][t], cls._indicators["LO_ENTRY"][t] = u_e, l_e
            cls._indicators["UP_EXIT"][t],  cls._indicators["LO_EXIT"][t]  = u_x, l_x
            cls._indicators["HURST"][t] = h_vals

        # 3. GENERATE RAW SIGNALS (🧬 2/3: Signal Logic)
        raw_signals = pd.DataFrame(0, index=data.index, columns=tickers)
        
        for t in tqdm.tqdm(tickers, desc="🧬 2/3: Signal Logic", leave=True):
            df_t = data[t]
            c_col = next(c for c in df_t.columns if c.lower() == 'close')
            
            # SPEED BOOST: Convert columns to NumPy arrays once per ticker
            prices   = df_t[c_col].values
            h_vals   = cls._indicators["HURST"][t].values
            up_entry = cls._indicators["UP_ENTRY"][t].values
            lo_entry = cls._indicators["LO_ENTRY"][t].values
            up_exit  = cls._indicators["UP_EXIT"][t].values
            lo_exit  = cls._indicators["LO_EXIT"][t].values
            
            sig_out = np.zeros(len(df_t))
            in_pos = False
            active_regime = None 
            
            start_idx = max(params["LOOKBACK"], params["HURST_WINDOW"]) + 1
            
            for i in range(start_idx, len(df_t)):
                # Using NumPy array indexing (i-1) is ultra-fast
                p_now = prices[i-1]
                h_now = h_vals[i-1]
                
                if not in_pos:
                    if h_now > params["HURST_TREND"]: # Trending
                        if p_now > up_entry[i-1]:
                            sig_out[i] = 1
                            in_pos, active_regime = True, "MOM"
                    elif h_now < params["HURST_REVERT"]: # Mean Reverting
                        if p_now < lo_entry[i-1]:
                            sig_out[i] = 1
                            in_pos, active_regime = True, "REV"
                else:
                    if active_regime == "MOM":
                        if p_now < lo_exit[i-1]:
                            sig_out[i] = -1
                            in_pos = False
                    elif active_regime == "REV":
                        if p_now > up_exit[i-1]:
                            sig_out[i] = -1
                            in_pos = False
            
            # Efficiently write the whole array back to the DataFrame column
            raw_signals[t] = sig_out

        cls._signals_df = raw_signals

        # 4. STORE RESULTS (🧬 3/3: Sticky Weights)
        with tqdm.tqdm(total=1, desc="🧬 3/3: Sticky Weights", leave=False) as pbar:
            cls._weights_df = signals_to_sticky_weights(raw_signals, params["MAX_STOCKS"])
            pbar.update(1)

    def get_signals(self, trading_state):
        ts = trading_state["current_timestamp"]
        return Strategy._weights_df.loc[ts], 0

In [7]:
class Backtester:
    def __init__(self, data: pd.DataFrame, initial_value: float, start_date):
        self.data = data
        self.initialvalue = initial_value
        self.portfolio_value = initial_value
        self.cash = initial_value
        self.investment = 0.0
        self.current_index = 1
        tickers = data.columns.get_level_values(0).unique()
        self.positions = pd.Series(0, index=tickers)
        self.all_positions = pd.DataFrame(columns=tickers)
        self.tradingState = {}
        self.all_signals = pd.DataFrame(columns=tickers)
        self.startdate = start_date

    def calculate_positions(self, signal: pd.Series, value, open=True) -> pd.Series:
        # --- Checks (Unchanged) ---
        if (signal < 0).any():
            raise ValueError(f'For timestamp {self.data.index[self.current_index]}, signal contains negative values: {signal[signal < 0]}')
        if not isinstance(signal, pd.Series):
            raise TypeError(f'For timestamp {self.data.index[self.current_index]}, signal must be a pandas Series, got {type(signal)}')
        if abs(signal).sum() - 1 > 1e-6:
            raise ValueError(f'For timestamp {self.data.index[self.current_index]} the sum of the abs(signals) must not be greater than 1, got {abs(signal).sum()}')

        prices = (
            self.data.xs('Open', level=1, axis=1).iloc[self.current_index]
            if open
            else self.data.xs('Close', level=1, axis=1).iloc[self.current_index]
        )
        prices = prices.reindex(signal.index)

        aligned_positions = self.positions.reindex(signal.index).fillna(0)

        nan_index = signal.isna()

        value -= (aligned_positions[nan_index] * prices[nan_index]).sum()

        float_shares = (signal.replace(0, np.nan) * value) / prices.replace(0, np.nan)
        float_shares = float_shares.replace([np.inf, -np.inf], 0).fillna(0)

        new_positions = pd.Series(0, index=float_shares.index, dtype=int)
        longs = float_shares > 0
        shorts = float_shares < 0

        new_positions[longs] = np.floor(float_shares[longs]).astype(int)
        new_positions[shorts] = np.ceil(float_shares[shorts]).astype(int)

        new_positions[nan_index] = aligned_positions[nan_index]

        return new_positions

    def calculate_cash(self, positions: pd.Series, open=True) -> float:
        index = self.current_index
        price = self.data.xs('Open', level=1, axis=1).iloc[index] if open else self.data.xs('Close', level=1, axis=1).iloc[index]
        return self.portfolio_value - (abs(positions) * price).sum()

    def update_investment(self, positions: pd.Series, new_day=False) -> float:
        index = self.current_index
        price1 = self.data.xs('Close', level=1, axis=1).iloc[index - 1] if new_day else self.data.xs('Open', level=1, axis=1).iloc[index]
        price2 = self.data.xs('Open', level=1, axis=1).iloc[index] if new_day else self.data.xs('Close', level=1, axis=1).iloc[index]
        return (positions * (price2 - price1)).sum() + self.investment

    def run(self, mode='normal'):
        output_signals_path = 'results/signals.csv'
        output_positions_path = 'results/positions.csv'

        import os
        if not os.path.exists('results'):
            os.makedirs('results')
            print("Created 'results' directory for output files.")
        
        if Strategy._weights_df is None:
                    # Fallback: if memory is empty, try to load from file (for backward compatibility)
                    weights_file = '.cache/backtester_weights_cache.csv' if mode == 'looptest' else 'auxilary/backtester_weights.csv'
                    if os.path.exists(weights_file):
                        Strategy._weights_df = pd.read_csv(weights_file, index_col=0, parse_dates=True)
                    else:
                        # If both memory AND file are missing, THEN raise the error
                        raise FileNotFoundError(f"Strategy weights not found in memory or at {weights_file}. "
                                            "Make sure Strategy.prepare() was called first.")

        print(f"Starting backtest (NumPy engine, mode='{mode}')...")
        if mode == 'looptest':
            strategy = Strategy_looptest()
        else:
            strategy = Strategy()

        tickers = self.data.columns.get_level_values(0).unique()
        n_assets = len(tickers)
        n_steps = len(self.data.index)

        open_df = self.data.xs('Open', level=1, axis=1).loc[:, tickers]
        close_df = self.data.xs('Close', level=1, axis=1).loc[:, tickers]
        open_px = open_df.to_numpy(dtype=float)
        close_px = close_df.to_numpy(dtype=float)

        pos_array = np.zeros(n_assets, dtype=np.int64)
        cash = float(self.initialvalue)

        all_pos_history = np.zeros((n_steps, n_assets), dtype=np.int64)
        all_sig_history = np.full((n_steps, n_assets), np.nan, dtype=float)

        traderData = 0
        # Inside Backtester.run:
        for i in tqdm.tqdm(range(0, n_steps), 
                        desc="⏳ Backtesting  ", 
                        position=2, 
                        leave=True, 
                        mininterval=0.5,      # Only update every 0.5 seconds
                        smoothing=0.1,        # Makes the growth look like a slider
                        bar_format='{l_bar}{bar:20}{r_bar}'):
            # ... logic ...
            ts = self.data.index[i]

            # Keep Strategy() contract: pull one weight row per step from the weights CSV
            self.tradingState = {
                'investment': self.investment,
                'cash': self.cash,
                'current_timestamp': ts,
                'traderData': traderData,
                'positions': self.positions,
            }
            signal, traderData = strategy.get_signals(self.tradingState)
            if signal is None:
                raise ValueError(f'For timestamp {ts}, signal is None')
            signal = signal.reindex(tickers)
            weights_row = signal.to_numpy(dtype=float)
            all_sig_history[i] = weights_row

            # --- Execution at Open[i], valuation at Close[i] ---
            curr_open = open_px[i]
            curr_close = close_px[i]

            # Mark-to-market at open before trading
            portfolio_value_open = cash + float(np.sum(pos_array * curr_open))

            update_mask = ~np.isnan(weights_row)
            if np.any(update_mask):
                # Validate long-only weights constraint (same idea as calculate_positions)
                if np.nansum(np.abs(weights_row)) - 1 > 1e-6:
                    raise ValueError(
                        f'For timestamp {ts} the sum of abs(weights) must not be greater than 1, got {np.nansum(np.abs(weights_row))}'
                    )
                if np.nanmin(weights_row) < -1e-12:
                    raise ValueError(f'For timestamp {ts}, weights contain negative values.')

                # Reserve value for holdings that are not being updated today (NaN => keep)
                reserved_value = float(np.sum(pos_array[~update_mask] * curr_open[~update_mask]))
                available_value = portfolio_value_open - reserved_value
                if available_value < 0:
                    available_value = 0.0

                target_weights = np.nan_to_num(weights_row[update_mask], nan=0.0)
                target_prices = curr_open[update_mask]

                # Allocate only the available value across updated tickers
                dollars = target_weights * available_value

                with np.errstate(divide='ignore', invalid='ignore'):
                    shares = np.floor(np.divide(dollars, target_prices, out=np.zeros_like(dollars), where=target_prices != 0.0))
                    shares = np.nan_to_num(shares, nan=0.0, posinf=0.0, neginf=0.0).astype(np.int64)

                pos_array[update_mask] = shares

            # Cash after trades at open
            cash = portfolio_value_open - float(np.sum(pos_array * curr_open))

            # End-of-day valuation at close
            investment_close = float(np.sum(pos_array * curr_close))
            portfolio_value_close = cash + investment_close

            all_pos_history[i] = pos_array
            self.current_index = i
            self.cash = cash
            self.investment = investment_close
            self.portfolio_value = portfolio_value_close

        # Store outputs as DataFrames (same shape as before)
        self.positions = pd.Series(pos_array, index=tickers)
        self.all_positions = pd.DataFrame(all_pos_history, index=self.data.index, columns=tickers)
        self.all_signals = pd.DataFrame(all_sig_history, index=self.data.index, columns=tickers)

        try:
            self.all_signals.to_csv(output_signals_path)
            self.all_positions.to_csv(output_positions_path)
            tqdm.tqdm.write(f"\n--- Checkpoint Saved progress to CSV files. ---")
        except Exception as e:
            tqdm.tqdm.write(f"\n--- Could not save checkpoint. Reason: {e} ---")

    def vectorbt_run(self, rolling_period=None, mean_period=None, rolling_threshold=None, dilute_to=None, mode='normal'):
            import os
            
            # Ensure start_date is handled
            if self.startdate is None or pd.isna(self.startdate):
                filtered_positions = self.all_positions
            else:
                filtered_positions = self.all_positions[self.all_positions.index >= self.startdate]
                if filtered_positions.empty:
                    filtered_positions = self.all_positions

            open_prices = self.data.xs('Open', level=1, axis=1).loc[filtered_positions.index, filtered_positions.columns]
            close_prices = self.data.xs('Close', level=1, axis=1).loc[filtered_positions.index, filtered_positions.columns]

            positions_to_use = filtered_positions
            order_size = positions_to_use.diff()
            order_size.iloc[0] = positions_to_use.iloc[0]
            order_size = order_size.astype(int)

            order_size = order_size.reindex(index=open_prices.index, columns=open_prices.columns).fillna(0)
            order_size = order_size.mask(order_size == 0)

            # 1. Run the VectorBT Portfolio
            portfolio = vbt.Portfolio.from_orders(
                close=close_prices,
                size=order_size,
                price=open_prices,
                init_cash=self.initialvalue,
                freq='1D',
                cash_sharing=True,
                call_seq='auto',
                log=True,
            )
            print(f"Initial Amount = {portfolio.init_cash}")

            # 2. Basic Stats
            stats_eq = portfolio.stats()
            stats_df = stats_eq.to_frame(name='Value').reset_index()
            stats_df.columns = ['Metric', 'Value']

            # 3. SAFER BENCHMARK LOGIC
            benchmark_path = 'src/benchmark.csv'
            # Check if benchmark file exists and required globals are defined
            if os.path.exists(benchmark_path) and 'START_DATE' in globals() and 'END_DATE' in globals():
                try:
                    benchmark_df = pd.read_csv(benchmark_path, index_col=0, parse_dates=True)
                    # Slice by globals
                    benchmark_df = benchmark_df.loc[START_DATE:END_DATE]
                    
                    if not benchmark_df.empty:
                        start_price = benchmark_df['Close'].iloc[0]
                        end_price = benchmark_df['Close'].iloc[-1]
                        benchmark_return = ((end_price - start_price) / start_price) * 100

                        # Clean existing benchmark placeholder if it exists
                        stats_df = stats_df[stats_df['Metric'] != 'Benchmark Return [%]'].reset_index(drop=True)
                        
                        benchmark_row = pd.DataFrame({
                            'Metric': ['Benchmark Return [%]'],
                            'Value': [benchmark_return]
                        })

                        # Insert benchmark return at a readable position (row 6)
                        stats_df = pd.concat([stats_df.iloc[:6], benchmark_row, stats_df.iloc[6:]], ignore_index=True)
                        print("Benchmark comparison added.")
                except Exception as e:
                    print(f"Warning: Could not process benchmark.csv: {e}")
            else:
                print("Notice: No benchmark.csv found or START_DATE/END_DATE not set. Skipping benchmark metrics.")

            # 4. CUSTOM PERIOD LOGIC
            custom_periods = [
                (None, '2020-03-23'),
                ('2020-03-23', '2021-10-18'),
                ('2021-10-18', '2023-03-27'),
                ('2023-03-27', '2024-09-24'),
                ('2024-09-24', '2025-09-30')
            ]

            period_metrics = []
            pf_value = portfolio.value()

            for start_str, end_str in custom_periods:
                try:
                    if start_str is None:
                        s_date = pf_value.index[0]
                        start_label = "Inception"
                    else:
                        s_date = pd.Timestamp(start_str)
                        start_label = start_str
                    
                    e_date = pd.Timestamp(end_str)
                    sub_period = pf_value[s_date:e_date]

                    if not sub_period.empty:
                        val_start = sub_period.iloc[0]
                        val_end = sub_period.iloc[-1]
                        period_ret = ((val_end - val_start) / val_start) * 100
                        period_metrics.append({'Metric': f'Return: {start_label} -> {end_str} [%]', 'Value': period_ret})
                except:
                    continue

            if period_metrics:
                stats_df = pd.concat([stats_df, pd.DataFrame(period_metrics)], ignore_index=True)

            # 5. Save results
            if not os.path.exists('hurst results'): os.makedirs('hurst results')
            portfolio.assets().to_csv('hurst results/assets.csv')
            portfolio.orders.records_readable.to_csv('hurst results/log.csv')
            
            df_out = pd.concat([portfolio.value(), portfolio.asset_value(), portfolio.cash()], axis=1)
            df_out.columns = ['portfolio', 'investment', 'cash']
            df_out.to_csv('hurst results/portfolio.csv')

            return portfolio, stats_df

In [8]:
# ================= CONFIG =================
INITIAL_CAPITAL = 1_000_000  # change if needed


In [9]:
# 1. Standardize column names (fixes 'open' vs 'Open')
# This handles the MultiIndex (Ticker, Price) structure
data.columns = data.columns.set_levels(
    [col.capitalize() for col in data.columns.levels[1]], 
    level=1
)

# 2. Verify the fix
print("Corrected columns:", data.columns.get_level_values(1).unique())
# You should see: Index(['Open', 'High', 'Low', 'Close', 'Volume', ...])

Corrected columns: Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')


In [12]:

# ============================================================
# 1. UTILITIES: ATOMIC SAVING & SILENT EXECUTION
# ============================================================

def save_atomic(df, path):
    """Overwrites the file atomically. One file to rule them all."""
    if not os.path.exists(os.path.dirname(path)) and os.path.dirname(path) != '':
        os.makedirs(os.path.dirname(path), exist_ok=True)
    
    tmp = path + ".tmp"
    df.to_csv(tmp)
    os.replace(tmp, path)

def append_csv(df, path):
    """Appends performance metrics to a master CSV."""
    write_header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=write_header, index=False)

def run_single_experiment(params, data, initial_capital=1000000):
    """Prepares and runs backtest while suppressing internal library prints."""
    Strategy.prepare(data, params)
    bt = Backtester(data, initial_capital, data.index[0])
    
    # Redirect stdout to devnull to hide "Initial Amount", "Notice", and progress bars
    with open(os.devnull, 'w') as devnull:
        with contextlib.redirect_stdout(devnull):
            bt.run(mode='normal')
            portfolio, stats_df = bt.vectorbt_run()
    
    # Process results
    metrics_dict = dict(zip(stats_df['Metric'], stats_df['Value']))
    perf_df = pd.DataFrame([{**params, **metrics_dict}])

    return perf_df, Strategy._weights_df, Strategy._signals_df

# ============================================================
# 2. CONFIGURATION & GRID SETUP
# ============================================================

# Define your parameter grid
param_grid = {
    "LOOKBACK": [ 20, 60, 180, 240],
    "EXIT_LOOKBACK": [ 20, 60, 180, 240],
    "MAX_STOCKS": [ 10, 12, 15, 20, 25, 30],
    "HURST_WINDOW": [2048,1024,512,256,128,64,16], # New: How much data to look at for regime
    "HURST_TREND": [0.5],           # New: Above this = Momentum
    "HURST_REVERT": [0.5]           # New: Below this = Mean Reversion
}

# File Paths
FOLDER = "hurst_results"
os.makedirs(FOLDER, exist_ok=True)
PERF_FILE = os.path.join(FOLDER, "hurst sweep performance.csv")
WEIGHTS_FILE = os.path.join(FOLDER, "hurst last successful weights.csv")
SIGNALS_FILE = os.path.join(FOLDER, "hurst last successful signals.csv")

# Setup Combinations
all_combinations = list(itertools.product(*param_grid.values()))
total_runs = len(all_combinations)
INITIAL_CAPITAL = 1000000

# ============================================================
# 3. THE CLEAN SWEEP LOOP
# ============================================================

print(f"🚀 Starting sweep: {total_runs} combinations")
print(f"📂 Results folder: {FOLDER}\n")

for run_idx, combo in enumerate(all_combinations, start=1):
    params = dict(zip(param_grid.keys(), combo))
    
    # Header for each run
    print(f"▶️ [{run_idx}/{total_runs}] Testing: {params}")

    try:
        # 1. Execute backtest silently
        perf_df, weights_df, signals_df = run_single_experiment(params, data, INITIAL_CAPITAL)

        # 2. Append stats & Overwrite latest weights/signals
        append_csv(perf_df, PERF_FILE)
        save_atomic(weights_df, WEIGHTS_FILE)
        save_atomic(signals_df, SIGNALS_FILE)
        
        # 3. Success message with clean spacing
        print(f"✅ Run {run_idx} completed and saved.")
        print("-" * 60) # Visual separator
        print()         # Breathing room

    except KeyboardInterrupt:
        print(f"\n🛑 User Interrupted. Files in /{FOLDER} reflect Run {run_idx-1}.")
        break
    except Exception as e:
        print(f"❌ Run {run_idx} failed with error: {e}")
        print("-" * 60 + "\n")

print("🏁 Sweep process finished.")

🚀 Starting sweep: 672 combinations
📂 Results folder: hurst_results

▶️ [1/672] Testing: {'LOOKBACK': 20, 'EXIT_LOOKBACK': 20, 'MAX_STOCKS': 10, 'HURST_WINDOW': 2048, 'HURST_TREND': 0.5, 'HURST_REVERT': 0.5}


🧬 2/3: Signal Logic: 100%|██████████| 496/496 [00:00<00:00, 1417.41it/s]
                                                                     
⏳ Backtesting  : 100%|████████████████████| 2231/2231 [00:00<00:00, 6376.60it/s]


✅ Run 1 completed and saved.
------------------------------------------------------------

▶️ [2/672] Testing: {'LOOKBACK': 20, 'EXIT_LOOKBACK': 20, 'MAX_STOCKS': 10, 'HURST_WINDOW': 1024, 'HURST_TREND': 0.5, 'HURST_REVERT': 0.5}


🧬 2/3: Signal Logic: 100%|██████████| 496/496 [00:00<00:00, 1019.19it/s]
                                                                     
⏳ Backtesting  : 100%|████████████████████| 2231/2231 [00:00<00:00, 6054.65it/s]


✅ Run 2 completed and saved.
------------------------------------------------------------

▶️ [3/672] Testing: {'LOOKBACK': 20, 'EXIT_LOOKBACK': 20, 'MAX_STOCKS': 10, 'HURST_WINDOW': 512, 'HURST_TREND': 0.5, 'HURST_REVERT': 0.5}


...

✅ Run 670 completed and saved.
------------------------------------------------------------

▶️ [671/672] Testing: {'LOOKBACK': 240, 'EXIT_LOOKBACK': 240, 'MAX_STOCKS': 30, 'HURST_WINDOW': 64, 'HURST_TREND': 0.5, 'HURST_REVERT': 0.5}


🧬 2/3: Signal Logic: 100%|██████████| 496/496 [00:00<00:00, 775.36it/s] 
                                                                     

⏳ Backtesting  : 100%|████████████████████| 2231/2231 [00:00<00:00, 4363.86it/s]


✅ Run 671 completed and saved.
------------------------------------------------------------

▶️ [672/672] Testing: {'LOOKBACK': 240, 'EXIT_LOOKBACK': 240, 'MAX_STOCKS': 30, 'HURST_WINDOW': 16, 'HURST_TREND': 0.5, 'HURST_REVERT': 0.5}


🧬 2/3: Signal Logic: 100%|██████████| 496/496 [00:00<00:00, 810.43it/s]
                                                                     
⏳ Backtesting  : 100%|████████████████████| 2231/2231 [00:00<00:00, 5626.10it/s]


✅ Run 672 completed and saved.
------------------------------------------------------------

🏁 Sweep process finished.
